In [ ]:
%%capture
import os
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
reports_folder = Path(os.environ["INTECOMM_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from django.contrib.sites.models import Site
from edc_sites.site import sites
from intecomm_analytics.dataframes import get_df_main_1858
from django_pandas.io import read_frame
from intecomm_reports.models import Diagnoses
from edc_constants.constants import FEMALE, MALE

In [ ]:
# use df_main to get conditions reported at screening
df_main_original = get_df_main_1858(None)
df_main = df_main_original.copy()

# use Diagnoses table (generated table) to get conditions confirmed at basline
dx_df = read_frame(Diagnoses.objects.all())


In [ ]:
# build a new dataset
# get demographics from df_main
# get conditions from dx_df
df = dx_df.merge(df_main[["subject_identifier", "gender", "age_in_years", "group_identifier"]], on="subject_identifier", how='left')

# flag for ncd cohort
df["ncd"] = ~df["hiv"]

# group 'group_identifier' on 'hiv', 'ncd'
grouped = df.groupby('group_identifier')[['hiv', 'ncd']].sum()
# calculate ratio HIV:NCD
grouped['ratio_hiv_ncd'] = grouped['hiv'] / grouped['ncd']
grouped["gte_0.5"] = grouped['ratio_hiv_ncd'] >= 0.5
# calculate # members per froup_identifier
grouped["members"] = grouped["hiv"] + grouped["ncd"]
grouped

In [ ]:
x=df_main[["subject_identifier", "gender", "age_in_years", "group_identifier"]].groupby(by=["group_identifier"]).describe()["age_in_years"]
type(x)

In [ ]:
grouped.ratio_hiv_ncd.describe()

In [ ]:
# number of groups


In [ ]:


def get_cells_for_categorical(df:pd.DataFrame, col:str, categories:list[str]|None=None, arm:str|None=None)->list[str]:
    if arm:
        n = len(df[(df['assignment']==arm) & (df[col].notna())])
        counts = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts()
        percentages = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts(normalize=True) * 100
    else:
        n = len(df[(df[col].notna())])
        counts = df[(df[col].notna())][col].value_counts()
        percentages = df[(df[col].notna())][col].value_counts(normalize=True) * 100
    cells = [n]
    for cat in categories:
        cells.append(f"{counts.get(cat, 0)} ({percentages.get(cat, 0):.1f}%)",)
    return cells

def get_cells_for_continuous(df, exclude_count:bool|None=None)->list[str]:
    cells = [] if exclude_count else [f"{int(df['count'])}"]
    cells.extend([f"{df['mean']:.2f}({df['std']:.2f})",
        f"{df['50%']:.2f}({df['min']:.2f}–{df['max']:.2f})"])
    return cells

def get_formatted_rows(df, col:str|None=None, **kwargs):
    """Returns 5 columns"""

    df = df[df[col].notna()].copy()
    df_all = df[col].describe()

    return  {
        'Statistics': ['Mean(sd)', 'Median(min-max)'],
        'All': [
            *get_cells_for_continuous(df_all, **kwargs),
        ],
    }

def get_formatted_rows_mf(df, col:str|None=None):
    """Returns 5 columns"""

    df = df[df[col].notna()].copy()
    df_all = df[col].describe()

    return  {
        'Statistics': ['n', 'Mean(sd)', 'Median(min-max)'],
        'All': [
            *get_cells_for_continuous(df_all),
        ],
        'Female': [
            *get_cells_for_continuous(df[df.gender==FEMALE][col].describe()),
        ],
        'Male': [
            *get_cells_for_continuous(df[df.gender==MALE][col].describe()),
        ],
    }

In [ ]:

dfs = []
table = {'Category': ["Groups"]}
table.update({
    'Parameter': [""],
    **{"Statistics": ["n"], "All": [f"{len(grouped)}"]}
})
table_df  = pd.DataFrame(table)
dfs.append(table_df.copy())

for label, param, col in [("Members per group","All", "members"), ("","HIV", "hiv"), ("", "NCD", "ncd"), ("Ratio HIV:NCD","", "ratio_hiv_ncd")]:
    table = {'Category': [label, '']}
    table.update({
        'Parameter': [param, ''],
        **get_formatted_rows(grouped, col, exclude_count=True)
    })
    table_df  = pd.DataFrame(table)
    dfs.append(table_df.copy())

table_dfs = pd.concat(dfs)
table_dfs

In [ ]:
from intecomm_group.models import PatientGroup

# there are 124 randomized groups
df = read_frame(PatientGroup.objects.filter(randomized=True))
df["site_id"] = df["site"].map({obj.domain: obj.id for obj in Site.objects.all()})
df["country"] = df["site_id"].apply(lambda x: sites.get(x).country.lower())
df = df.drop(columns=["site"])
df = df.reset_index(drop=True)


In [ ]:
df[["site_id", "group_identifier", "randomized_datetime", "status", "ratio", "bypass_group_size_min", "bypass_group_ratio"]]

In [ ]:
df["country"].value_counts().sort_index()

In [ ]:
df["site_id"].value_counts().sort_index()

In [ ]:
df["site_id"].value_counts().sort_index().describe()

In [ ]:
df["site_id"].value_counts().sort_index().to_frame().reset_index().plot.box(column="count")

In [ ]:
df.groupby(["site_id", "country"], group_keys=True).size().to_frame().reset_index()["country"].value_counts()

we randomized 124 groups between two countries (TZ=59, UG=65) over 14 sites (TZ=6, UG=8)
with an average 9 groups per site (8.8, +/-2.4)(9, 5-13)


In [ ]:

df["site_id"].value_counts().sort_index().to_frame().reset_index().plot.box(column="count")


In [ ]:
# subjects per group
df_dx = get_diagnoses_df()
df1 = df_dx.groupby(["country", "site_id", "group_identifier"]).size().to_frame().reset_index()
df1 = df1.rename(columns={0:"subjects"})
df1["subjects"].describe()

In [ ]:
df2 = df1[df1.country=="tanzania"]["subjects"].describe().to_frame().reset_index()
df2 = df2.rename(columns={"subjects":"group", "index":"stats"})
df2["country"] = "tanzania"
df2

In [ ]:
df3 = df1[df1.country=="uganda"]["subjects"].describe().to_frame().reset_index()
df3 = df3.rename(columns={"subjects":"group", "index":"stats"})
df3["country"] = "uganda"
df3

In [ ]:
df4 = df1["subjects"].describe().to_frame().reset_index()
df4 = df4.rename(columns={"subjects":"group", "index":"stats"})
df4["country"] = "both"
df4

In [ ]:
df5 = pd.concat([df2,df3, df4])

In [ ]:
df6 = df5.pivot_table(index=["country"], columns=["stats"], values=["group"])
df6

In [ ]:
df6.to_csv("group.csv")